<a href="https://colab.research.google.com/github/ArjunBhakta/Data-Science-Cohort-20/blob/main/Project_3_Chinook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Project 3 - Chinook


### Load Chinook data set



In [1]:
# Install the sqlite package for Ubuntu
# Download the Chinook sqlite database
import sqlite3 as db
import pandas as pd
from google.colab import output

In [2]:
%%capture
%%bash
apt-get update
apt-get install -y sqlite3
pip install sqlite-web


In [3]:
%%bash
[ -f chinook.zip ] ||
  curl -s -O https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip
unzip -l chinook.zip


Archive:  chinook.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
   884736  2015-11-29 10:53   chinook.db
---------                     -------
   884736                     1 file


In [4]:
!rm -f chinook.db


In [5]:
!unzip -u chinook.zip

Archive:  chinook.zip
  inflating: chinook.db              


In [6]:
# Get a list of the tables in the database
%%script sqlite3 --column --header chinook.db
.tables


albums          employees       invoices        playlists     
artists         genres          media_types     tracks        
customers       invoice_items   playlist_track


In [7]:
# Show the schema for the entire database
%%script sqlite3 --column --header chinook.db
.schema

CREATE TABLE IF NOT EXISTS "albums"
(
    [AlbumId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Title] NVARCHAR(160)  NOT NULL,
    [ArtistId] INTEGER  NOT NULL,
    FOREIGN KEY ([ArtistId]) REFERENCES "artists" ([ArtistId]) 
		ON DELETE NO ACTION ON UPDATE NO ACTION
);
CREATE TABLE sqlite_sequence(name,seq);
CREATE TABLE IF NOT EXISTS "artists"
(
    [ArtistId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Name] NVARCHAR(120)
);
CREATE TABLE IF NOT EXISTS "customers"
(
    [CustomerId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [FirstName] NVARCHAR(40)  NOT NULL,
    [LastName] NVARCHAR(20)  NOT NULL,
    [Company] NVARCHAR(80),
    [Address] NVARCHAR(70),
    [City] NVARCHAR(40),
    [State] NVARCHAR(40),
    [Country] NVARCHAR(40),
    [PostalCode] NVARCHAR(10),
    [Phone] NVARCHAR(24),
    [Fax] NVARCHAR(24),
    [Email] NVARCHAR(60)  NOT NULL,
    [SupportRepId] INTEGER,
    FOREIGN KEY ([SupportRepId]) REFERENCES "employees" ([EmployeeId]) 
		ON DELETE NO ACTION ON 


## Approach: Persona-Driven Analytics

Instead of analyzing the dataset from a single perspective, this project approaches the music business through the lens of five distinct business personas.  

Each persona represents a different stakeholder inside a company — from marketing and sales to finance, product, and customer retention. The goal is to simulate how real organizations ask questions of their data and how analytics can support decision-making across teams.

By framing the analysis around personas, the project becomes less about writing isolated SQL queries and more about solving meaningful business problems.

### Why Personas?

Different roles care about different outcomes:

- Marketing teams care about growth opportunities and audience targeting.
- Sales managers care about customer ownership and performance.
- Finance analysts focus on revenue trends and business health.
- Product managers look for engagement signals and usage patterns.
- CX / Retention teams identify churn risk and customer loyalty.

This creates a more realistic analytics workflow because the same dataset can answer fundamentally different questions depending on who is asking.

---

### Personas + Core Questions

| Persona | Business Goal | Core Question |
|---|---|---|
| Marketing Lead | Identify growth opportunities | Which countries and genres represent the biggest untapped revenue opportunity for the next campaign? |
| Sales Manager | Understand rep performance | Which support reps manage the highest-value customers and what behaviors separate top performers? |
| Finance Analyst | Monitor business health | What are the key drivers behind month-over-month revenue changes? |
| Product Manager | Improve engagement | Which genres, artists, and playlist behaviors are strongest signals of repeat purchases? |
| CX / Retention | Reduce churn | Which customers are most at risk of churning based on purchase recency and declining spend? |

---

### Analytical Philosophy

This project emphasizes:

- Translating business questions into data questions
- Connecting SQL analysis to operational decisions
- Thinking cross-functionally about the same dataset
- Designing analytics around stakeholders instead of tables
- Building intuition for how organizations use data in practice

The intention is to be less about "query writing" and move toward developing a product and business-oriented analytics mindset.

## 1. Marketing Lead

### Business Goal
Identify the strongest market opportunities by understanding which countries and music genres generate the highest engagement and revenue.

### Core Question
Which countries and genres represent the biggest untapped revenue opportunity for the next marketing campaign?

### Why This Matters
This helps marketing teams decide:
- where to focus campaigns
- which genres to promote
- which markets have the highest growth potential

In [8]:
%%script sqlite3 --column --header chinook.db

SELECT
    c.Country,
    g.Name AS Genre,
    COUNT(DISTINCT c.CustomerId) AS NumberOfCustomers,
    COUNT(ii.InvoiceLineId) AS TracksPurchased,
    ROUND(SUM(ii.UnitPrice * ii.Quantity), 2) AS TotalRevenue

FROM customers c
JOIN invoices i
    ON c.CustomerId = i.CustomerId
JOIN invoice_items ii
    ON i.InvoiceId = ii.InvoiceId
JOIN tracks t
    ON ii.TrackId = t.TrackId
JOIN genres g
    ON t.GenreId = g.GenreId
GROUP BY
    c.Country,
    g.Name
HAVING
    COUNT(ii.InvoiceLineId) > 20
ORDER BY
    TotalRevenue DESC;

Country         Genre               NumberOfCustomers  TracksPurchased  TotalRevenue
--------------  ------------------  -----------------  ---------------  ------------
USA             Rock                13                 157              155.43      
Canada          Rock                8                  107              105.93      
USA             Latin               13                 91               90.09       
Brazil          Rock                5                  81               80.19       
France          Rock                5                  65               64.35       
USA             Metal               13                 64               63.36       
Germany         Rock                4                  62               61.38       
Canada          Latin               7                  60               59.4        
Brazil          Latin               5                  53               52.47       
USA             Alternative & Punk  11                 50        

## 2. Sales Manager

### Business Goal
Understand which support representatives manage the most valuable customer relationships.

### Core Question
Which support reps manage the highest-value customers and what separates top performers from the rest?

### Why This Matters
This helps identify:
- top-performing reps
- customer concentration
- revenue contribution by employee
- high-value customer ownership

In [9]:
%%script sqlite3 --column --header chinook.db

SELECT
    e.EmployeeId,
    e.FirstName || ' ' || e.LastName AS SupportRep,
    COUNT(DISTINCT c.CustomerId) AS CustomersManaged,
    COUNT(DISTINCT i.InvoiceId) AS NumberOfInvoices,
    ROUND(SUM(i.Total), 2) AS TotalRevenue,
    ROUND(AVG(i.Total), 2) AS AvgInvoiceValue
FROM employees e
JOIN customers c
    ON e.EmployeeId = c.SupportRepId
JOIN invoices i
    ON c.CustomerId = i.CustomerId
GROUP BY
    e.EmployeeId,
    SupportRep
ORDER BY
    TotalRevenue DESC;

EmployeeId  SupportRep     CustomersManaged  NumberOfInvoices  TotalRevenue  AvgInvoiceValue
----------  -------------  ----------------  ----------------  ------------  ---------------
3           Jane Peacock   21                146               833.04        5.71           
4           Margaret Park  20                140               775.4         5.54           
5           Steve Johnson  18                126               720.16        5.72           


## 3. Finance Analyst

### Business Goal
Track financial performance and understand business momentum over time.

### Core Question
What are the key drivers behind month-over-month revenue changes?

### Why This Matters
This helps finance teams:
- monitor growth
- detect slowdowns
- identify seasonal patterns
- measure overall business health

In [10]:
%%script sqlite3 --column --header chinook.db

WITH monthly_revenue AS (
    SELECT
        strftime('%Y-%m', InvoiceDate) AS RevenueMonth,
        COUNT(InvoiceId) AS NumberOfInvoices,
        ROUND(SUM(Total), 2) AS MonthlyRevenue,
        ROUND(AVG(Total), 2) AS AvgInvoiceValue
    FROM invoices
    GROUP BY RevenueMonth
)

SELECT
    RevenueMonth,
    NumberOfInvoices,
    MonthlyRevenue,
    AvgInvoiceValue,
    ROUND(
        AVG(MonthlyRevenue) OVER (
            ORDER BY RevenueMonth
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ),

        2 ) AS MovingAverage3Month

FROM monthly_revenue

ORDER BY RevenueMonth;

RevenueMonth  NumberOfInvoices  MonthlyRevenue  AvgInvoiceValue  MovingAverage3Month
------------  ----------------  --------------  ---------------  -------------------
2009-01       6                 35.64           5.94             35.64              
2009-02       7                 37.62           5.37             36.63              
2009-03       7                 37.62           5.37             36.96              
2009-04       7                 37.62           5.37             37.62              
2009-05       7                 37.62           5.37             37.62              
2009-06       7                 37.62           5.37             37.62              
2009-07       7                 37.62           5.37             37.62              
2009-08       7                 37.62           5.37             37.62              
2009-09       7                 37.62           5.37             37.62              
2009-10       7                 37.62           5.37             

## 4. Product Manager

### Business Goal
Understand which music content drives repeat engagement and purchasing behavior.

### Core Question
Which genres, artists, and albums are strongest signals of repeat purchases and long-term engagement?

### Why This Matters
This helps product teams:
- identify high-performing catalog content
- understand engagement signals
- improve recommendation systems
- drive user retention

In [11]:
%%script sqlite3 --column --header chinook.db
SELECT
    g.Name AS Genre,
    ar.Name AS Artist,
    al.Title AS Album,
    COUNT(ii.InvoiceLineId) AS PurchaseCount,
    COUNT(DISTINCT i.CustomerId) AS UniqueCustomers,
    ROUND(SUM(ii.UnitPrice * ii.Quantity), 2) AS TotalRevenue

FROM invoice_items ii
JOIN invoices i
    ON ii.InvoiceId = i.InvoiceId
JOIN tracks t
    ON ii.TrackId = t.TrackId
JOIN genres g
    ON t.GenreId = g.GenreId
JOIN albums al
    ON t.AlbumId = al.AlbumId
JOIN artists ar
    ON al.ArtistId = ar.ArtistId
GROUP BY
    g.Name,
    ar.Name,
    al.Title
ORDER BY
    PurchaseCount DESC
LIMIT 20


Genre               Artist                          Album                                                             PurchaseCount  UniqueCustomers  TotalRevenue
------------------  ------------------------------  ----------------------------------------------------------------  -------------  ---------------  ------------
Latin               Chico Buarque                   Minha Historia                                                    27             11               26.73       
Alternative & Punk  Titãs                           Acústico                                                          22             8                21.78       
Rock                Kiss                            Greatest Kiss                                                     20             7                19.8        
Latin               Caetano Veloso                  Prenda Minha                                                      19             7                18.81       
Rock                Cr

## 5. CX / Retention

### Business Goal
Identify high-value customers who may be at risk of churning based on declining engagement.

### Core Question
Which high-value customers have not purchased recently?

### Why This Matters
This helps retention teams:
- identify churn risk
- prioritize customer outreach
- protect high-value accounts
- improve customer lifetime value retention

In [12]:
%%script sqlite3 --column --header chinook.db
SELECT
    c.CustomerId,
    c.FirstName || ' ' || c.LastName AS CustomerName,
    c.Country,
    COUNT(i.InvoiceId) AS TotalPurchases,
    ROUND(SUM(i.Total), 2) AS LifetimeValue,
    MAX(i.InvoiceDate) AS LastPurchaseDate,
    ROUND(julianday((SELECT MAX(InvoiceDate) FROM invoices)) - julianday(MAX(i.InvoiceDate)), 0) AS DaysSinceLastPurchase
FROM customers c
JOIN invoices i
    ON c.CustomerId = i.CustomerId
GROUP BY
    c.CustomerId,
    CustomerName,
    c.Country
HAVING
    LifetimeValue > 40
    AND DaysSinceLastPurchase > 90
ORDER BY
    LifetimeValue DESC;

CustomerId  CustomerName           Country         TotalPurchases  LifetimeValue  LastPurchaseDate     DaysSinceLastPurchase
----------  ---------------------  --------------  --------------  -------------  -------------------  ---------------------
26          Richard Cunningham     USA             7               47.62          2013-04-05 00:00:00  261.0                
57          Luis Rojas             Chile           7               46.62          2012-10-14 00:00:00  434.0                
45          Ladislav Kovács        Hungary         7               45.62          2013-07-20 00:00:00  155.0                
24          Frank Ralston          USA             7               43.62          2013-08-20 00:00:00  124.0                
28          Julia Barnett          USA             7               43.62          2013-05-19 00:00:00  217.0                
37          Fynn Zimmermann        Germany         7               43.62          2013-06-03 00:00:00  202.0                
